# 01. Build Card Features

Build one row per card from the transaction data. The feature set follows the EDA findings from `EDA/03_card_level_behavior.ipynb`.

In [27]:
from pathlib import Path

import numpy as np
import pandas as pd

from mdq.data import PROJECT_ROOT, load_data

pd.options.display.float_format = "{:,.4f}".format

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_COUNTRY = "Kazakhstan"

In [28]:
business, consumer, merchants, mcc = load_data()

business.shape, consumer.shape, merchants.shape, mcc.shape

((2997593, 12), (9832487, 12), (2165, 5), (1303, 4))

In [29]:
B2B_MCC_GROUPS = {
    "advertising": ["7311"],
    "software_cloud": ["7372"],
    "business_services": ["7399", "7392"],
    "office_supplies": ["5943", "5111"],
    "telecom": ["4814", "4816"],
    "logistics": ["4214", "4215", "4225"],
    "professional_services": ["8111", "8931", "8999"],
}


def enrich_transactions(transactions, merchants, mcc):
    out = transactions.copy()
    out["mcc"] = out["mcc"].astype(str)

    merchants_no_mcc = merchants.drop(columns=["mcc"]).copy()
    out = out.merge(merchants_no_mcc, on="merchant_id", how="left")

    mcc_ref = mcc[["mcc", "mcc_name"]].copy()
    mcc_ref["mcc"] = mcc_ref["mcc"].astype(str)
    out = out.merge(mcc_ref, on="mcc", how="left")

    out["merchant_key"] = out["merchant_id"]
    mask = (out["merchant_id"] == "MER_000000") & (out["mcc"] == "7012")
    out.loc[mask, "merchant_key"] = "MER_000000_MCC_7012"

    out["ts"] = pd.to_datetime(out["transaction_timestamp"])
    out["hour"] = out["ts"].dt.hour
    out["date"] = out["ts"].dt.date
    out["month"] = out["ts"].dt.to_period("M")

    out["is_online"] = out["channel"].str.lower().eq("online")
    out["is_pos"] = out["channel"].str.lower().eq("pos")
    out["is_weekend"] = out["ts"].dt.dayofweek >= 5
    out["weekday_business_hours"] = (~out["is_weekend"]) & out["hour"].between(9, 18)
    out["weekday_non_business_hours"] = (~out["is_weekend"]) & ~out["hour"].between(9, 18)
    out["is_evening"] = out["hour"].between(18, 21)
    out["is_night"] = out["hour"].ge(22) | out["hour"].le(5)
    out["foreign_transaction"] = out["country"].ne(LOCAL_COUNTRY)
    out["foreign_merchant"] = out["merchant_country"].ne(LOCAL_COUNTRY)

    return out


business_txn = enrich_transactions(business, merchants, mcc)
consumer_txn = enrich_transactions(consumer, merchants, mcc)

business_txn.shape, consumer_txn.shape

((2997593, 30), (9832487, 30))

In [30]:
def concentration_features(df, column, prefix):
    counts = df.groupby(["card_number", column]).size()
    totals = counts.groupby(level=0).sum()
    probs = counts / totals

    amounts = df.groupby(["card_number", column])["transaction_amount_kzt"].sum()
    amount_totals = amounts.groupby(level=0).sum()
    amount_probs = amounts / amount_totals

    return pd.DataFrame({
        f"unique_{prefix}": counts.groupby(level=0).size(),
        f"{prefix}_entropy": (-(probs * np.log(probs))).groupby(level=0).sum(),
        f"{prefix}_hhi": (probs ** 2).groupby(level=0).sum(),
        f"top_{prefix}_share": probs.groupby(level=0).max(),
        f"top_{prefix}_amount_share": amount_probs.groupby(level=0).max(),
    })


def monthly_growth(df):
    monthly = (
        df.groupby(["card_number", "month"])["transaction_amount_kzt"]
        .sum()
        .reset_index()
    )

    months = sorted(monthly["month"].unique())
    early_months = months[:3]
    late_months = months[-3:]

    early = monthly[monthly["month"].isin(early_months)].groupby("card_number")["transaction_amount_kzt"].mean()
    late = monthly[monthly["month"].isin(late_months)].groupby("card_number")["transaction_amount_kzt"].mean()

    return ((late - early) / early.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan).fillna(0)


def build_card_features(df, segment, label):
    g = df.groupby("card_number")
    features = pd.DataFrame(index=g.size().index)
    features.index.name = "card_number"

    amount = g["transaction_amount_kzt"]
    features["txn_count"] = g.size()
    features["total_spend"] = amount.sum()
    features["avg_amount"] = amount.mean()
    features["median_amount"] = amount.median()
    features["std_amount"] = amount.std().fillna(0)
    features["p90_amount"] = amount.quantile(0.90)
    features["amount_cv"] = (features["std_amount"] / features["avg_amount"]).replace([np.inf, -np.inf], np.nan).fillna(0)

    for col in ["total_spend", "avg_amount", "median_amount", "std_amount", "p90_amount"]:
        features[f"log_{col}"] = np.log1p(features[col])

    features["active_days"] = g["date"].nunique()
    features["txn_per_active_day"] = features["txn_count"] / features["active_days"]
    features["amount_per_active_day"] = features["total_spend"] / features["active_days"]
    features["log_amount_per_active_day"] = np.log1p(features["amount_per_active_day"])

    for threshold in [50_000, 100_000, 500_000]:
        features[f"large_txn_{threshold}_ratio"] = g["transaction_amount_kzt"].apply(lambda s, t=threshold: s.ge(t).mean())

    ratio_cols = [
        "is_online",
        "is_recurring",
        "tokenized",
        "recurring_capable",
        "weekday_business_hours",
        "is_weekend",
        "is_evening",
        "is_night",
        "foreign_transaction",
        "foreign_merchant",
    ]

    ratios = g[ratio_cols].mean().rename(columns={
        "is_online": "online_ratio",
        "is_recurring": "recurring_ratio",
        "tokenized": "tokenized_ratio",
        "recurring_capable": "recurring_capable_ratio",
        "weekday_business_hours": "weekday_business_hours_ratio",
        "is_weekend": "weekend_ratio",
        "is_evening": "evening_ratio",
        "is_night": "night_ratio",
        "foreign_transaction": "foreign_transaction_ratio",
        "foreign_merchant": "foreign_merchant_ratio",
    })
    features = features.join(ratios)

    features["unique_transaction_countries"] = g["country"].nunique()
    features["unique_merchant_countries"] = g["merchant_country"].nunique()

    features = features.join(concentration_features(df, "merchant_key", "merchant"))
    features = features.rename(columns={"unique_merchant": "unique_merchants"})
    features = features.join(concentration_features(df, "mcc", "mcc"))

    for group, codes in B2B_MCC_GROUPS.items():
        features[f"mcc_{group}_ratio"] = g["mcc"].apply(lambda s, c=codes: s.isin(c).mean())
        amount_in_group = df["transaction_amount_kzt"].where(df["mcc"].isin(codes), 0)
        features[f"mcc_{group}_amount_ratio"] = amount_in_group.groupby(df["card_number"]).sum() / features["total_spend"]

    b2b_codes = sorted({code for codes in B2B_MCC_GROUPS.values() for code in codes})
    features["mcc_b2b_total_ratio"] = g["mcc"].apply(lambda s: s.isin(b2b_codes).mean())
    b2b_amount = df["transaction_amount_kzt"].where(df["mcc"].isin(b2b_codes), 0)
    features["mcc_b2b_total_amount_ratio"] = b2b_amount.groupby(df["card_number"]).sum() / features["total_spend"]

    raw_money_cols = [
        "total_spend",
        "avg_amount",
        "median_amount",
        "std_amount",
        "p90_amount",
        "amount_per_active_day",
    ]
    features = features.drop(columns=raw_money_cols)

    features = features.reset_index()
    features.insert(1, "segment", segment)
    features.insert(2, "label", label)

    return features


business_features = build_card_features(business_txn, "business", 1)
consumer_features = build_card_features(consumer_txn, "consumer", 0)
card_features = pd.concat([business_features, consumer_features], ignore_index=True)

business_features.shape, consumer_features.shape, card_features.shape

((25000, 54), (80000, 54), (105000, 54))

In [31]:
card_features.head()

,card_number,segment,label,txn_count,amount_cv,log_total_spend,log_avg_amount,log_median_amount,log_std_amount,log_p90_amount,...,mcc_office_supplies_ratio,mcc_office_supplies_amount_ratio,mcc_telecom_ratio,mcc_telecom_amount_ratio,mcc_logistics_ratio,mcc_logistics_amount_ratio,mcc_professional_services_ratio,mcc_professional_services_amount_ratio,mcc_b2b_total_ratio,mcc_b2b_total_amount_ratio
0,5100610003025081,business,1,178,1.7037,16.0826,10.9009,9.5819,11.4337,12.0933,...,0.0000,0.0000,0.0506,0.0022,0.0337,0.0245,0.0899,0.2422,0.2809,0.5945
1,5100610003044611,business,1,106,1.4692,16.4795,11.8161,11.1245,12.2008,12.7507,...,0.0189,0.0292,0.2170,0.0087,0.3491,0.2713,0.0094,0.0045,0.7358,0.6167
2,5100610003860784,business,1,148,1.1499,16.3883,11.3911,10.9410,11.5308,12.3555,...,0.0405,0.0984,0.0000,0.0000,0.6081,0.4440,0.0000,0.0000,0.7635,0.7651
3,5100610008756482,business,1,196,1.1714,17.1239,11.8458,11.4964,12.0040,12.7451,...,0.0204,0.0481,0.0459,0.0528,0.0306,0.0097,0.0000,0.0000,0.2143,0.2571
4,5100610013466473,business,1,120,1.3517,16.6152,11.8277,10.7577,12.1291,12.8292,...,0.0167,0.0010,0.0750,0.1481,0.1667,0.0486,0.0000,0.0000,0.7667,0.8940


In [32]:
business_features.to_parquet(PROCESSED_DIR / "business_card_features.parquet", index=False, engine="fastparquet")
consumer_features.to_parquet(PROCESSED_DIR / "consumer_card_features.parquet", index=False, engine="fastparquet")
card_features.to_parquet(PROCESSED_DIR / "card_features.parquet", index=False, engine="fastparquet")

PROCESSED_DIR

PosixPath('/Users/sapuantalaspay/vs_projects/data/data/processed')

In [33]:
card_features.columns

Index(['card_number', 'segment', 'label', 'txn_count', 'amount_cv',
       'log_total_spend', 'log_avg_amount', 'log_median_amount',
       'log_std_amount', 'log_p90_amount', 'active_days', 'txn_per_active_day',
       'log_amount_per_active_day', 'large_txn_50000_ratio',
       'large_txn_100000_ratio', 'large_txn_500000_ratio', 'online_ratio',
       'recurring_ratio', 'tokenized_ratio', 'recurring_capable_ratio',
       'weekday_business_hours_ratio', 'weekend_ratio', 'evening_ratio',
       'night_ratio', 'foreign_transaction_ratio', 'foreign_merchant_ratio',
       'unique_transaction_countries', 'unique_merchant_countries',
       'unique_merchants', 'merchant_entropy', 'merchant_hhi',
       'top_merchant_share', 'top_merchant_amount_share', 'unique_mcc',
       'mcc_entropy', 'mcc_hhi', 'top_mcc_share', 'top_mcc_amount_share',
       'mcc_advertising_ratio', 'mcc_advertising_amount_ratio',
       'mcc_software_cloud_ratio', 'mcc_software_cloud_amount_ratio',
       'mcc_busine